# 🏢 مسكن ثلاثي الأبعاد: مسار المعالجة والتدريب عبر Kaggle (T4 GPU)
### Maskan 3D: Kaggle-Native Reconstruction & Training Pipeline

هذا الدفتر مخصص لتشغيل خط المعالجة والتدريب لنماذج **3D Gaussian Splatting** في بيئة **Kaggle** (باستخدام بطاقة T4 ومسار `/kaggle/working`).
- **المدخل:** فيديو جولة مشي أو دوران بانورامي من Google Drive أو مرفوع مباشرة كـ Dataset.
- **المعالجة:** استخراج الإطارات الذكي، والتصفية من التشويش والتكرار، وحساب مواضع الكاميرات عبر COLMAP بالـ CPU (متوافق تجارياً 100%).
- **التدريب:** تدريب نموذج Splatfacto لـ 15,000 خطوة بدرجة توافقيات صفرية (SH=0) لإنتاج مجسم خفيف وعالي الأداء للموبايل.

## ⚙️ 1. إعداد المتغيرات والمسارات الرئيسية (Configuration & Paths)
قم بضبط معرّف ملف الفيديو `DRIVE_FILE_ID` من رابط Google Drive، أو اترك المسار الافتراضي إذا رفعت الفيديو مباشرة إلى Kaggle.

In [ ]:
# ========================================================
# ⚙️ إعدادات تجربة مسكن 3D على Kaggle (Maskan 3D Configuration)
# ========================================================
import os
from pathlib import Path

# 1. معرّف ملف الفيديو أو رابط المجلد على Google Drive
DRIVE_FILE_ID = "1Q-n4eVCTGWL8SMwsva1Lk9uMH7rL2hef"  # معرّف ملف الفيديو مباشرة
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1wZa2kZkUSTKE9LFCijYBUsFhXRP7ONLw"  # رابط المجلد كاملاً

# 2. مسارات العمل في Kaggle
OUT_ROOT = "/kaggle/working"
VIDEO_PATH = "/kaggle/working/video.mp4"

# 3. معايير استخراج الإطارات والمعالجة
FPS_EXTRACT = 2            # معدل استخراج الإطارات في الثانية من الفيديو
MAX_FRAMES = 250           # الحد الأقصى للإطارات المختارة للمعالجة
MATCHING = "sequential"    # طريقة المطابقة (sequential مخصصة للفيديوهات المتسلسلة)
RELAXED_MAPPER = False     # مسار احتياطي لتخفيف شروط التثليث في COLMAP إذا فشل الافتراضي

# 4. معايير تدريب Gaussian Splatting
SH_DEGREE = 0              # درجة التوافقيات الكروية (0 لحجم خفيف < 25 MB وسرعة فائقة على الموبايل)
ITERATIONS = 15000         # عدد خطوات التدريب (15000 خطوة متوازنة للمشاهد الداخلية)

# المسارات المشتقة المنظمة
EXTRACTED_RAW_DIR = os.path.join(OUT_ROOT, "extracted_raw")
FILTERED_FRAMES_DIR = os.path.join(OUT_ROOT, "filtered_frames")
PROCESSED_DATA_DIR = os.path.join(OUT_ROOT, "data/nerfstudio/apartment")
TRAIN_OUTPUTS_DIR = os.path.join(OUT_ROOT, "outputs")
EXPERIMENT_NAME = "apartment_splat"
DIAGNOSTICS_DIR = os.path.join(OUT_ROOT, "diagnostics")
RESULTS_DIR = os.path.join(OUT_ROOT, "results")
ZIP_OUTPUT_PATH = os.path.join(OUT_ROOT, "maskan3d_results.zip")

# ملفات السجلات
PROCESS_LOG_PATH = os.path.join(OUT_ROOT, "process.log")
TRAIN_LOG_PATH = os.path.join(OUT_ROOT, "train.log")

print("✅ تم ضبط الإعدادات والمسارات بنجاح.")
print(f"📁 مجلد العمل الرئيسي: {OUT_ROOT}")


## 🔍 2. فحص بيئة التشغيل والاتصال (Environment & Network Check)
التحقق من توفر بطاقة الرسوميات GPU وإصدار بايثون والتأكد من تفعيل الاتصال بالإنترنت في Kaggle.

In [ ]:
import sys
import urllib.request

print("=" * 60)
print("🔍 فحص بيئة التشغيل في Kaggle (Environment Check):")
print(f"🐍 إصدار بايثون (Python Version): {sys.version}")
print("=" * 60)

# 1. فحص مواصفات كارت الشاشة GPU
!nvidia-smi

# 2. فحص الاتصال بالإنترنت
try:
    urllib.request.urlopen("https://www.google.com", timeout=5)
    print("\n🌐 الاتصال بالإنترنت: متصل بنجاح (Internet is ON ✅)")
except Exception as e:
    print("\n" + "!" * 70)
    print("⚠️ تنبيه حرج: الإنترنت غير مفعّل في إعدادات نوت بوك Kaggle!")
    print("⚠️ لن تتمكن من تثبيت الحزم أو تنزيل الفيديو.")
    print("👉 الحل: من القائمة الجانبية اليمنى (Settings)، قم بتفعيل: Internet -> Internet on")
    print("!" * 70 + "\n")


## 📦 3. تثبيت المتطلبات البرمجية وتدقيق التراخيص (Install Dependencies)

> [!IMPORTANT]
> **التدقيق التجاري القانوني لـ COLMAP (Commercial Licence Notice):**
> كود COLMAP الأساسي مرخص بـ BSD-3-Clause، ولكن تفعيل تسريع كارت الشاشة لـ SIFT يستدعي مكتبة `SiftGPU` المحمية برخصة بحثية غير تجارية صريحة (`repos/colmap/src/thirdparty/SiftGPU/LICENSE`).
> لضمان الامتثال التجاري الكامل 100% في مسكن 3D، نعتمد على تشغيل COLMAP عبر الـ CPU فقط المعتمد على خوارزميات BSD أو OpenCV المتساهلة تجارياً عبر علم `--no-gpu`.

In [ ]:
# 1. تثبيت أدوات النظام عبر apt (COLMAP بالـ CPU و FFmpeg لمعالجة الفيديو)
!apt-get update -y && apt-get install -y colmap ffmpeg

# 2. تثبيت nerfstudio و gdown
!pip install --upgrade gdown nerfstudio


## 📥 4. تنزيل الفيديو وفحص خصائصه (Download Video & Metadata)
تنزيل الفيديو من Google Drive بواسطة أداة `gdown`، أو التوجيه لرفعه كـ Dataset إذا تعذر التنزيل.

In [ ]:
import os
import glob
import shutil
import subprocess
from pathlib import Path

download_success = False

# 1. محاولة التنزيل المباشر لمعرّف ملف الفيديو
if DRIVE_FILE_ID.strip():
    print(f"📥 1. محاولة تنزيل الفيديو مباشرة من معرّف الملف (ID: {DRIVE_FILE_ID})...")
    url = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"
    res = os.system(f"gdown --fuzzy '{url}' -O '{VIDEO_PATH}'")
    if res == 0 and os.path.exists(VIDEO_PATH) and os.path.getsize(VIDEO_PATH) > 1024:
        download_success = True
        print(f"✅ تم تنزيل الفيديو بنجاح إلى: {VIDEO_PATH}")

# 2. إذا فشل التنزيل المباشر، تنزيل المجلد بالكامل واستخراج ملف الفيديو منه
if not download_success and 'DRIVE_FOLDER_URL' in globals() and DRIVE_FOLDER_URL.strip():
    print(f"📥 2. جاري تنزيل المجلد بالكامل من Google Drive لاستخراج الفيديو...")
    drive_tmp = os.path.join(OUT_ROOT, "drive_download")
    os.makedirs(drive_tmp, exist_ok=True)
    os.system(f"gdown --folder '{DRIVE_FOLDER_URL}' -O '{drive_tmp}'")
    video_candidates = []
    for ext in ["*.mp4", "*.mov", "*.mkv", "*.avi"]:
        video_candidates.extend(glob.glob(f"{drive_tmp}/**/{ext}", recursive=True))
    if video_candidates:
        found_video = sorted(video_candidates, key=os.path.getsize, reverse=True)[0]
        shutil.copy(found_video, VIDEO_PATH)
        download_success = True
        print(f"✅ تم اكتشاف واستخراج ملف الفيديو: {os.path.basename(found_video)}")
        print(f"📁 تم حفظ الفيديو إلى: {VIDEO_PATH}")

if not download_success:
    if os.path.exists(VIDEO_PATH) and os.path.getsize(VIDEO_PATH) > 1024:
        print(f"ℹ️ استخدام ملف الفيديو الموجود مسبقاً في: {VIDEO_PATH}")
        download_success = True
    else:
        print("\n" + "=" * 70)
        print("⚠️ تعذر تنزيل الفيديو تلقائياً عبر Google Drive!")
        print("💡 [مسار بديل مجاني وسريع في Kaggle]:")
        print("1. ارفع الفيديو مباشرة في Kaggle عبر الزر: (+ Add Input) ثم (Upload Dataset).")
        print("2. انسخ مسار الفيديو من القائمة اليمنى (Input)، مثل:")
        print("   /kaggle/input/my-apartment-video/video.mp4")
        print("3. عدّل متغير VIDEO_PATH في خلية الإعدادات الأولى إلى هذا المسار وأعد تشغيل الخلية.")
        print("=" * 70 + "\n")

# فحص حجم ومدة الفيديو عبر ffprobe
if download_success:
    size_mb = os.path.getsize(VIDEO_PATH) / (1024 * 1024)
    print(f"📦 حجم ملف الفيديو: {size_mb:.2f} MB")
    try:
        probe_cmd = f"ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 '{VIDEO_PATH}'"
        duration_sec = float(subprocess.check_output(probe_cmd, shell=True).decode().strip())
        print(f"⏱️ مدة الفيديو: {duration_sec:.1f} ثانية ({duration_sec/60:.2f} دقيقة)")
    except Exception as e:
        print(f"⚠️ تعذر قراءة مدة الفيديو: {e}")


## 🎞️ 5. استخراج الإطارات وتصفية التشويش والتكرار (Frame Extraction & Quality Filtering)
يتم استخراج الإطارات بـ FFmpeg ثم تصفيتها هندسياً:
1. **استبعاد الإطارات المكررة أو الثابتة (Near-Duplicate Rejection):** لحماية COLMAP من الدوران البطيء في المكان.
2. **استبعاد الإطارات الضبابية (Blur Filtering):** بحساب تباين لابلاسيان وتطبيق معيار Z-Score القوي (Robust-Z).
3. **توزيع زمني منتظم (Time-Window Uniform Sampling):** لضمان عدم تجاوز `MAX_FRAMES` مع اختيار اللقطة الأحدّ في كل نافذة.

In [ ]:
import os
import glob
import shutil
import cv2
import numpy as np
from pathlib import Path

# تهيئة المجلدات
os.makedirs(EXTRACTED_RAW_DIR, exist_ok=True)
os.makedirs(FILTERED_FRAMES_DIR, exist_ok=True)

# تنظيف المجلدات السابقة
for f in glob.glob(os.path.join(EXTRACTED_RAW_DIR, "*")):
    os.remove(f)
for f in glob.glob(os.path.join(FILTERED_FRAMES_DIR, "*")):
    os.remove(f)

print(f"🎞️ 1. استخراج الإطارات بمعدل {FPS_EXTRACT} إطار/ثانية عبر FFmpeg...")
extract_cmd = f'ffmpeg -y -i "{VIDEO_PATH}" -vf fps={FPS_EXTRACT} -q:v 2 "{EXTRACTED_RAW_DIR}/frame_%05d.png"'
os.system(extract_cmd)

raw_frames = sorted(glob.glob(os.path.join(EXTRACTED_RAW_DIR, "frame_*.png")))
num_raw = len(raw_frames)
print(f"📸 إجمالي الإطارات المستخرجة خام: {num_raw} إطار")

if num_raw == 0:
    raise RuntimeError(f"❌ لم يتم استخراج أي إطارات من الفيديو: {VIDEO_PATH}")

# 2. حساب تباين لابلاسيان (Laplacian Variance) ومؤشر التشابه بين الإطارات المتتالية
variances = []
kept_after_dup = []
dropped_dup_count = 0
prev_gray_small = None
DUPLICATE_DIFF_THRESH = 4.0  # عتبة الفرق بين الإطارات المتتالية لكشف الثبات

print("🔍 2. فحص التشويش (Blur) والتكرار بين الإطارات المتتالية...")
for frame_path in raw_frames:
    img = cv2.imread(frame_path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # فحص التكرار (Near-duplicate check)
    gray_small = cv2.resize(gray, (128, 128))
    if prev_gray_small is not None:
        diff = np.mean(np.abs(gray_small.astype(np.float32) - prev_gray_small.astype(np.float32)))
        if diff < DUPLICATE_DIFF_THRESH:
            dropped_dup_count += 1
            continue  # تخطي الإطار المتكرر
    prev_gray_small = gray_small
    
    var = cv2.Laplacian(gray, cv2.CV_64F).var()
    variances.append(var)
    kept_after_dup.append((frame_path, var))

print(f"🗑️ إطارات مكررة/شبه متطابقة تم إسقاطها: {dropped_dup_count}")

# 3. تصفية الإطارات الضبابية باستخدام Robust Z-Score (Median Absolute Deviation)
vars_arr = np.array([v for _, v in kept_after_dup])
med_var = np.median(vars_arr)
mad_var = np.median(np.abs(vars_arr - med_var))
robust_z = (vars_arr - med_var) / (1.4826 * mad_var + 1e-6)

ROBUST_Z_THRESH = -2.0
sharp_frames = []
dropped_blur_count = 0

for (path, v), z in zip(kept_after_dup, robust_z):
    if z >= ROBUST_Z_THRESH:
        sharp_frames.append((path, v))
    else:
        dropped_blur_count += 1

print(f"🌫️ إطارات ضبابية (Blurry) تم إسقاطها: {dropped_blur_count}")
print(f"✨ الإطارات الحادة المتبقية: {len(sharp_frames)}")

# 4. تقسيم الإطارات إلى نوافذ زمنية لاختيار الأحدّ منها بحيث لا يتجاوز الإجمالي MAX_FRAMES
if len(sharp_frames) > MAX_FRAMES:
    print(f"✂️ تقليص الإطارات إلى الحد الأقصى المطلوب ({MAX_FRAMES}) عبر النوافذ الزمنية...")
    indices = np.array_split(np.arange(len(sharp_frames)), MAX_FRAMES)
    final_selected = []
    for chunk in indices:
        if len(chunk) == 0:
            continue
        best_idx = chunk[np.argmax([sharp_frames[i][1] for i in chunk])]
        final_selected.append(sharp_frames[best_idx][0])
else:
    final_selected = [p for p, _ in sharp_frames]

# نسخ الإطارات المختارة نهائياً
for i, src_path in enumerate(final_selected):
    dst_name = f"frame_{i+1:05d}.png"
    shutil.copy(src_path, os.path.join(FILTERED_FRAMES_DIR, dst_name))

print("=" * 60)
print("📊 التقرير النهائي لتصفية الإطارات:")
print(f"  • الإطارات الخام المستخرجة: {num_raw}")
print(f"  • تم إسقاطها لتكرار المحتوى: {dropped_dup_count}")
print(f"  • تم إسقاطها لعدم الحدة (Blur): {dropped_blur_count}")
print(f"  • الإطارات المعتمدة للمعالجة: {len(final_selected)} إطار")
print(f"  • مسار حفظ الإطارات: {FILTERED_FRAMES_DIR}")
print("=" * 60)


## 📐 6. استخراج الكاميرات ومصفوفات التثليث عبر COLMAP القياسي (Process Images)
تشغيل `ns-process-data images` بالمطابقة المتسلسلة (`--matching-method sequential`) وبدون GPU (`--no-gpu`) لحظر SiftGPU تجارياً.

In [ ]:
import os
import shutil
from pathlib import Path

# تنظيف المخرجات السابقة لضمان تشغيل نظيف
if os.path.exists(PROCESSED_DATA_DIR):
    shutil.rmtree(PROCESSED_DATA_DIR)
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print("🚀 بدء استخراج الكاميرات وحساب مصفوفات التثليث عبر COLMAP...")
print(f"📁 مجلد المدخلات: {FILTERED_FRAMES_DIR}")
print(f"📁 مجلد المخرجات: {PROCESSED_DATA_DIR}")
print(f"⚙️ طريقة المطابقة: {MATCHING} (sequential)")
print(f"⚙️ المعالجة: CPU فقط (--no-gpu) للامتثال التجاري")

# تشغيل أمر المعالجة مع توجيه وحفظ السجل عبر tee
cmd_process = (
    f"ns-process-data images "
    f"--data \"{FILTERED_FRAMES_DIR}\" "
    f"--output-dir \"{PROCESSED_DATA_DIR}\" "
    f"--matching-method {MATCHING} "
    f"--camera-type perspective "
    f"--no-gpu "
    f"2>&1 | tee \"{PROCESS_LOG_PATH}\""
)

print(f"أمر التشغيل:\n{cmd_process}\n")
!{cmd_process}


## 🛠️ 7. مسار COLMAP اليدوي المخفف كاحتياطي (Relaxed Mapper Fallback)
يعمل هذا المسار فقط في حال تم ضبط `RELAXED_MAPPER = True` في خلية الإعدادات الأولى.
يستخدم المعايير المخففة: `--Mapper.init_min_tri_angle 3.0` و `--Mapper.min_num_matches 8` و `--Mapper.abs_pose_min_num_inliers 8`.

> [!NOTE]
> التحقق من مصدر Nerfstudio يوضح أنه لا يوجد أمر CLI مباشر باسم `ns-process-data colmap`، بل يتم التحويل إما عبر دالة بايثون `colmap_to_json` المدمجة أو عبر أمر `ns-process-data images --skip-colmap`.

In [ ]:
import os
import shutil
from pathlib import Path

if RELAXED_MAPPER:
    print("⚠️ تفعيل مسار COLMAP اليدوي المخفف (Relaxed Mapper Fallback)...")
    colmap_dir = Path(PROCESSED_DATA_DIR) / "colmap"
    colmap_dir.mkdir(parents=True, exist_ok=True)
    db_path = colmap_dir / "database.db"
    sparse_dir = colmap_dir / "sparse"
    sparse_dir.mkdir(parents=True, exist_ok=True)
    
    print("1️⃣ استخراج المعالم (feature_extractor) عبر الـ CPU...")
    cmd_feat = (
        f"colmap feature_extractor "
        f"--database_path \"{db_path}\" "
        f"--image_path \"{FILTERED_FRAMES_DIR}\" "
        f"--ImageReader.single_camera 1 "
        f"--ImageReader.camera_model OPENCV "
        f"--SiftExtraction.use_gpu 0"
    )
    !{cmd_feat}
    
    print("2️⃣ المطابقة المتسلسلة (sequential_matcher) عبر الـ CPU...")
    cmd_match = (
        f"colmap sequential_matcher "
        f"--database_path \"{db_path}\" "
        f"--SiftMatching.use_gpu 0"
    )
    !{cmd_match}
    
    print("3️⃣ بناء النموذج (mapper) بالمعايير المخففة للتثليث...")
    cmd_map = (
        f"colmap mapper "
        f"--database_path \"{db_path}\" "
        f"--image_path \"{FILTERED_FRAMES_DIR}\" "
        f"--output_path \"{sparse_dir}\" "
        f"--Mapper.init_min_tri_angle 3.0 "
        f"--Mapper.min_num_matches 8 "
        f"--Mapper.abs_pose_min_num_inliers 8"
    )
    !{cmd_map}
    
    print("4️⃣ تحويل نموذج COLMAP إلى صيغة transforms.json...")
    try:
        from nerfstudio.process_data.colmap_utils import colmap_to_json
        sub_models = [d for d in sparse_dir.iterdir() if d.is_dir()]
        if not sub_models:
            raise RuntimeError("لم يتم العثور على أي نموذج sparse تم توليده!")
        target_model = sorted(sub_models, key=lambda d: len(list(d.glob("*.bin"))), reverse=True)[0]
        num_matched = colmap_to_json(
            recon_dir=target_model,
            output_dir=Path(PROCESSED_DATA_DIR)
        )
        print(f"✅ تم توليد transforms.json بنجاح لكاميرات عددها: {num_matched}")
    except Exception as e:
        print(f"⚠️ فشل التحويل عبر دالة colmap_to_json: {e}")
        print("محاولة التحويل عبر أمر ns-process-data images --skip-colmap...")
        cmd_skip = (
            f"ns-process-data images "
            f"--data \"{FILTERED_FRAMES_DIR}\" "
            f"--output-dir \"{PROCESSED_DATA_DIR}\" "
            f"--skip-colmap --no-gpu"
        )
        !{cmd_skip}
else:
    print("ℹ️ مسار RELAXED_MAPPER غير مفعّل. تم الاعتماد على نتائج المعالجة القياسية في الخلية السابقة.")


## 📐 8. تشخيص هندسة الكاميرات ومسار الحركة (Camera Geometry Diagnostics)
خلية تشخيصية حاسمة تقرأ ملف `transforms.json` فقط وبواسطة مكتبة `numpy` فقط:
- تقيس أبعاد الصندوق المحيط وطول المسار وتشتت الإزاحة مقابل الدوران.
- تكشف عدد المحطات (Stations) وتحدد هل النمط بانورامي بالدوران في المكان أم انتقال حقيقي (`PANORAMA-LIKE` vs `TRANSLATING`).
- تحفظ مخططاً علوياً (Top-down PNG) لمواضع الكاميرات وأسهم التوجيه داخل مجلد `diagnostics/`.

In [ ]:
import os
import json
import glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

os.makedirs(DIAGNOSTICS_DIR, exist_ok=True)
transforms_path = os.path.join(PROCESSED_DATA_DIR, "transforms.json")

print("=" * 65)
print("📐 تشخيص هندسة الكاميرات ومسار الحركة (Camera Geometry Diagnostics)")
print("=" * 65)

if not os.path.exists(transforms_path):
    print("❌ تعذر العثور على ملف transforms.json! يبدو أن معالجة COLMAP لم تسجل أي كاميرات.")
    reg_pct = 0.0
else:
    with open(transforms_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
        
    frames = meta.get("frames", [])
    num_registered = len(frames)
    input_frames = glob.glob(os.path.join(FILTERED_FRAMES_DIR, "*.png"))
    num_input = len(input_frames)
    reg_pct = (num_registered / max(num_input, 1)) * 100.0
    
    print(f"📸 1. نسبة تسجيل الكاميرات:")
    print(f"   • الإطارات المسجلة: {num_registered} من أصل {num_input} ({reg_pct:.1f}%)")
    
    if num_registered >= 2:
        positions = []
        view_dirs = []
        for frame in frames:
            c2w = np.array(frame["transform_matrix"], dtype=np.float64)
            pos = c2w[:3, 3]
            v_dir = c2w[:3, 2]
            v_dir = v_dir / (np.linalg.norm(v_dir) + 1e-8)
            positions.append(pos)
            view_dirs.append(v_dir)
            
        positions = np.array(positions)
        view_dirs = np.array(view_dirs)
        
        # 2. حدود الصندوق المحيط وطول المسار
        min_pos = np.min(positions, axis=0)
        max_pos = np.max(positions, axis=0)
        extents = max_pos - min_pos
        diag = np.linalg.norm(extents)
        diffs = np.diff(positions, axis=0)
        path_length = np.sum(np.linalg.norm(diffs, axis=1))
        
        print(f"\n📦 2. أبعاد الصندوق المحيط وطول المسار (وحدات اعتباطية):")
        print(f"   • اتساع الصندوق [X, Y, Z]: [{extents[0]:.3f}, {extents[1]:.3f}, {extents[2]:.3f}]")
        print(f"   • القطر الإجمالي للصندوق: {diag:.3f}")
        print(f"   • إجمالي طول مسار الكاميرا: {path_length:.3f}")
        
        # 3. المسافات بين أقرب الكاميرات (Nearest-Neighbor)
        dists = np.linalg.norm(positions[:, None, :] - positions[None, :, :], axis=-1)
        np.fill_diagonal(dists, np.inf)
        nn_dists = np.min(dists, axis=1)
        median_nn = np.median(nn_dists)
        max_nn = np.max(nn_dists)
        
        print(f"\n📏 3. المسافات بين أقرب كاميرا متجاورة (Nearest-Neighbor):")
        print(f"   • وسيط المسافة (Median NN): {median_nn:.4f}")
        print(f"   • أقصى مسافة انفصال (Max NN): {max_nn:.4f}")
        
        # 4. تشتت الإزاحة مقابل تشتت الدوران
        std_pos = np.mean(np.std(positions, axis=0))
        std_rot = np.mean(np.std(view_dirs, axis=0))
        ratio_trans_rot = std_pos / (std_rot + 1e-8)
        
        print(f"\n🔄 4. تشتت الإزاحة (Translation) مقابل الدوران (Rotation):")
        print(f"   • تشتت المواضع (Position Std): {std_pos:.4f}")
        print(f"   • تشتت متجهات الرؤية (Rotation Std): {std_rot:.4f}")
        print(f"   • النسبة (Translation/Rotation Ratio): {ratio_trans_rot:.4f}")
        
        # 5. التجميع العنقودي للكاميرات لتحديد المحطات (Station Clustering)
        cluster_thresh = 0.12 * diag
        clusters = []
        cluster_centers = []
        for idx, pos in enumerate(positions):
            if len(clusters) == 0:
                clusters.append([idx])
                cluster_centers.append(pos)
            else:
                d_to_centers = [np.linalg.norm(pos - c) for c in cluster_centers]
                min_c_idx = np.argmin(d_to_centers)
                if d_to_centers[min_c_idx] <= cluster_thresh:
                    clusters[min_c_idx].append(idx)
                    cluster_centers[min_c_idx] = np.mean(positions[clusters[min_c_idx]], axis=0)
                else:
                    clusters.append([idx])
                    cluster_centers.append(pos)
                    
        cluster_sizes = [len(c) for c in clusters]
        sorted_clusters = sorted(cluster_sizes, reverse=True)
        
        print(f"\n📍 5. تحليل محطات التصوير (Stations & Clusters):")
        print(f"   • عدد المحطات المكتشفة: {len(clusters)}")
        print(f"   • أحجام العناقيد (أكبر 5 محطات): {sorted_clusters[:5]}")
        
        # 6. الحكم الهندسي النهائي (Verdict)
        top3_coverage = sum(sorted_clusters[:3]) / num_registered
        if top3_coverage >= 0.70 or ratio_trans_rot < 0.15:
            verdict = "PANORAMA-LIKE (دوران في المكان - تغطية زوايا مع خط أساس ضيق)"
            verdict_code = "PANORAMA-LIKE"
        else:
            verdict = "TRANSLATING (حركة انتقال حقيقية بخط أساس واسع)"
            verdict_code = "TRANSLATING"
            
        print("\n" + "-" * 65)
        print(f"⚖️ الحكم الهندسي على نمط الالتقاط: [{verdict_code}]")
        print(f"   {verdict}")
        print("-" * 65)
        
        # 7. رسم وحفظ المخطط العلوي لمسار الكاميرات
        plt.figure(figsize=(8, 8))
        plt.scatter(positions[:, 0], positions[:, 1], c=np.arange(num_registered), cmap="viridis", s=40, label="Camera Centers")
        plt.quiver(positions[:, 0], positions[:, 1], view_dirs[:, 0], view_dirs[:, 1], color="red", alpha=0.6, scale=25, label="Viewing Directions")
        for c_idx, c_pos in enumerate(cluster_centers):
            plt.plot(c_pos[0], c_pos[1], "kx", markersize=10, markeredgewidth=2)
            
        plt.title(f"Camera Trajectory (Top-Down) - Verdict: {verdict_code}", fontsize=12)
        plt.xlabel("X (arbitrary units)")
        plt.ylabel("Y (arbitrary units)")
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend(loc="upper right")
        plt.tight_layout()
        
        diag_plot_path = os.path.join(DIAGNOSTICS_DIR, "camera_trajectory_topdown.png")
        plt.savefig(diag_plot_path, dpi=150)
        plt.close()
        print(f"💾 تم حفظ المخطط العلوي لمسار الكاميرات إلى:\n   {diag_plot_path}")


## 🚦 9. بوابة التحقق من نسبة التسجيل (Registration Gate)
- نسبة $\ge 90\%$: متابعة التدريب (ممتاز).
- نسبة $70\% - 89\%$: تحذير ومتابعة التدريب.
- نسبة $< 70\%$: إيقاف التنفيذ إجبارياً لحماية موارد الحساب وتنبيه المستخدم لتعديل التصوير أو تفعيل الـ Relaxed Mapper.

In [ ]:
# التحقق من نسبة التسجيل المحسوبة من خلية التشخيص السابقة
if 'reg_pct' not in globals():
    raise RuntimeError("❌ لم يتم حساب نسبة التسجيل! تأكد من تشغيل خلية التشخيص السابقة بنجاح.")

print(f"📊 فحص بوابة نسبة تسجيل الكاميرات: النسبة المحسوبة = {reg_pct:.1f}%")

if reg_pct >= 90.0:
    print("✅ ممتاز: نسبة التسجيل مستوفاة للشروط (>= 90%). تم فتح البوابة للمتابعة.")
    REGISTRATION_PASSED = True
elif reg_pct >= 70.0:
    print("⚠️ تحذير: نسبة التسجيل متوسطة (70% - 89%). المشهد سيتدرب ولكن قد تظهر فراغات طفيفة.")
    REGISTRATION_PASSED = True
else:
    REGISTRATION_PASSED = False
    print("!" * 70)
    print(f"❌ توقف إلزامي (Blocked): نسبة التسجيل {reg_pct:.1f}% أقل من الحد الأدنى المقبول (70%)!")
    print("💡 لا يمكن تدريب نموذج عالي الجودة بنسبة تسجيل منخفضة.")
    print("💡 جرب تفعيل RELAXED_MAPPER = True في خلية الإعدادات لإعادة التثليث بمعايير مخففة.")
    print("!" * 70)
    raise RuntimeError(f"Registration rate {reg_pct:.1f}% is below 70% threshold. Execution halted.")


## 🚀 10. تدريب نموذج 3D Gaussian Splatting (Train Splatfacto)
تدريب النموذج لـ `ITERATIONS` خطوة مع ضبط التوافقيات الصفرية (`--pipeline.model.sh-degree 0`) وإلغاء فتح المستعرض التفاعلي لضمان إنهاء الجلسة تلقائياً.

In [ ]:
import os
from pathlib import Path

# التحقق من اجتياز البوابة
if not globals().get('REGISTRATION_PASSED', False):
    raise RuntimeError("❌ لا يمكن بدء التدريب لأن بوابة التسجيل لم تجتز بنجاح.")

# تثبيت متغيرات بيئة كاجل للسرعة
os.environ["MAX_JOBS"] = "4"

print("=" * 65)
print(f"🚀 بدء تدريب Splatfacto لعدد {ITERATIONS} خطوة...")
print(f"🎨 درجة التوافقيات الكروية: SH_DEGREE = {SH_DEGREE} (لحجم ملف خفيف وسريع الفرز)")
print(f"📁 مسار حفظ المخرجات: {TRAIN_OUTPUTS_DIR}")
print(f"📝 توجيه السجل إلى: {TRAIN_LOG_PATH}")
print("=" * 65)

# تشغيل التدريب بدون المستعرض التفاعلي وحفظ المخرجات في السجل
cmd_train = (
    f"MAX_JOBS=4 ns-train splatfacto "
    f"--data \"{PROCESSED_DATA_DIR}\" "
    f"--output-dir \"{TRAIN_OUTPUTS_DIR}\" "
    f"--experiment-name \"{EXPERIMENT_NAME}\" "
    f"--max-num-iterations {ITERATIONS} "
    f"--pipeline.model.sh-degree {SH_DEGREE} "
    f"--steps-per-save 1000 "
    f"--vis tensorboard "
    f"--viewer.quit-on-train-completion True "
    f"2>&1 | tee \"{TRAIN_LOG_PATH}\""
)

print(f"أمر التشغيل:\n{cmd_train}\n")
!{cmd_train}


## 📦 11. تصدير المجسم بصيغة .ply وفحص عدد الجسيمات (Export Gaussian Splat)
تصدير النموذج المدرب بصيغة `.ply`، وقراءة الهيدر البرمجي فقط لاستخراج عدد الجسيمات بدقة وحساب حجم الملف.

In [ ]:
import os
import glob
from pathlib import Path

# البحث عن ملف config.yml الأحدث الناتج من التدريب
config_files = glob.glob(f"{TRAIN_OUTPUTS_DIR}/{EXPERIMENT_NAME}/splatfacto/*/config.yml")
if not config_files:
    raise RuntimeError(f"❌ لم يتم العثور على config.yml في: {TRAIN_OUTPUTS_DIR}/{EXPERIMENT_NAME}/splatfacto/")

latest_config = sorted(config_files, key=os.path.getmtime, reverse=True)[0]
print(f"✅ سيتم التصدير باستخدام الإعدادات: {latest_config}")

EXPORT_DIR = os.path.join(OUT_ROOT, "exports")
os.makedirs(EXPORT_DIR, exist_ok=True)
EXPORT_PLY_PATH = os.path.join(EXPORT_DIR, f"{EXPERIMENT_NAME}.ply")

# معالجة توافقية PyTorch 2.6 weights_only إذا تطلب الأمر
patch_code = """import torch\norig_load = torch.load\ndef safe_load(*args, **kwargs):\n    kwargs['weights_only'] = False\n    return orig_load(*args, **kwargs)\ntorch.load = safe_load\n"""
with open("/tmp/patch_torch.py", "w") as f:
    f.write(patch_code)

print("📦 جاري تصدير المجسم بصيغة .ply...")
export_cmd = f"python -c \"import sys; sys.path.insert(0, '/tmp'); import patch_torch; from nerfstudio.scripts.exporter import entrypoint; entrypoint()\" gaussian-splat --load-config '{latest_config}' --output-dir '{EXPORT_DIR}'"
res = os.system(export_cmd)

# إذا تم تصدير الملف باسم splat.ply القياسي، نعيد تسميته
default_ply = os.path.join(EXPORT_DIR, "splat.ply")
if os.path.exists(default_ply) and not os.path.exists(EXPORT_PLY_PATH):
    os.rename(default_ply, EXPORT_PLY_PATH)

# قراءة هيدر الـ PLY فقط لحساب عدد الجسيمات وحجم الملف
if os.path.exists(EXPORT_PLY_PATH):
    size_mb = os.path.getsize(EXPORT_PLY_PATH) / (1024 * 1024)
    num_gaussians = 0
    with open(EXPORT_PLY_PATH, "rb") as f:
        for _ in range(50):
            line = f.readline().decode("ascii", errors="ignore").strip()
            if line.startswith("element vertex"):
                num_gaussians = int(line.split()[-1])
            elif line == "end_header":
                break
                
    print("=" * 60)
    print("🎉 تم تصدير مجسم الـ Gaussian Splat بنجاح!")
    print(f"  • مسار الملف: {EXPORT_PLY_PATH}")
    print(f"  • حجم الملف: {size_mb:.2f} MB")
    print(f"  • إجمالي عدد الجسيمات الغاوسية: {num_gaussians:,} جسيم")
    print("=" * 60)
else:
    print(f"⚠️ تعذر العثور على ملف الـ PLY المصدّر في: {EXPORT_PLY_PATH}")


## 📥 12. تجميع النتائج وضغط الحزمة للتنزيل (Package & Export Zip)
تجميع كافة الملفات الهامة في أرشيف واحد `maskan3d_results.zip`:
1. مجسم الـ `.ply` النهائي.
2. ملف الكاميرات `transforms.json`.
3. نموذج كولماب المتفرق `colmap/sparse/0`.
4. سجلات المعالجة والتدريب (`process.log` و `train.log`).
5. مخطط التشخيص الهندسي `camera_trajectory_topdown.png`.
6. ملف الإعدادات `config.yml`.

In [ ]:
import os
import glob
import shutil
import zipfile
from pathlib import Path

os.makedirs(RESULTS_DIR, exist_ok=True)
print("📦 جاري تجميع وتسليم حزمة النتائج (Packaging Results)...")

# 1. نسخ مجسم الـ PLY
if os.path.exists(EXPORT_PLY_PATH):
    shutil.copy(EXPORT_PLY_PATH, os.path.join(RESULTS_DIR, os.path.basename(EXPORT_PLY_PATH)))
    print("  • تم نسخ مجسم الـ PLY")

# 2. نسخ transforms.json
transforms_file = os.path.join(PROCESSED_DATA_DIR, "transforms.json")
if os.path.exists(transforms_file):
    shutil.copy(transforms_file, os.path.join(RESULTS_DIR, "transforms.json"))
    print("  • تم نسخ transforms.json")

# 3. نسخ مجلد نقاط كولماب المتفرقة (colmap/sparse/0)
sparse_src = os.path.join(PROCESSED_DATA_DIR, "colmap", "sparse", "0")
if not os.path.exists(sparse_src):
    candidates = glob.glob(f"{PROCESSED_DATA_DIR}/colmap/**/cameras.bin", recursive=True)
    if candidates:
        sparse_src = os.path.dirname(candidates[0])

if os.path.exists(sparse_src):
    dst_sparse = os.path.join(RESULTS_DIR, "colmap_sparse_0")
    if os.path.exists(dst_sparse):
        shutil.rmtree(dst_sparse)
    shutil.copytree(sparse_src, dst_sparse)
    print(f"  • تم نسخ مجلد نموذج كولماب المتفرق: {sparse_src}")

# 4. نسخ سجلات المعالجة والتدريب
for log_name, log_path in [("process.log", PROCESS_LOG_PATH), ("train.log", TRAIN_LOG_PATH)]:
    if os.path.exists(log_path):
        shutil.copy(log_path, os.path.join(RESULTS_DIR, log_name))
        print(f"  • تم نسخ سجل {log_name}")

# 5. نسخ مخطط التشخيص الهندسي
diag_png = os.path.join(DIAGNOSTICS_DIR, "camera_trajectory_topdown.png")
if os.path.exists(diag_png):
    shutil.copy(diag_png, os.path.join(RESULTS_DIR, "camera_trajectory_topdown.png"))
    print("  • تم نسخ مخطط تشخيص مسار الكاميرات")

# 6. نسخ config.yml
if 'latest_config' in globals() and os.path.exists(latest_config):
    shutil.copy(latest_config, os.path.join(RESULTS_DIR, "config.yml"))
    print("  • تم نسخ config.yml")

# 7. إنشاء الأرشيف المضغوط
with zipfile.ZipFile(ZIP_OUTPUT_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(RESULTS_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, RESULTS_DIR)
            zipf.write(full_path, rel_path)

zip_size_mb = os.path.getsize(ZIP_OUTPUT_PATH) / (1024 * 1024)

print("\n" + "=" * 65)
print("🎉 اكتمل إعداد حزمة النتائج النهائية بنجاح!")
print(f"📦 مسار الأرشيف: {ZIP_OUTPUT_PATH}")
print(f"📊 حجم الأرشيف: {zip_size_mb:.2f} MB")
print("=" * 65)
print("\n📥 [تعليمات التنزيل المباشر من Kaggle]:")
print("1. من الشريط الجانبي الأيمن في Kaggle، انقر على أيقونة التبويب: (Output).")
print("2. ستجد ملف: maskan3d_results.zip داخل المجلد /kaggle/working.")
print("3. اضغط على النقاط الثلاث (...) بجوار الملف واختر: (Download).")
print("=" * 65 + "\n")
